# 07. 수기 정답셋 기반 OCR·필드 파이프라인 평가

입력은 현재 PDF 5개를 새로 OCR한 OpenAI Vision·Upstage 원문이다. `gold/raw`는 전체 OCR 평가의 기준, `gold/structured`는 필드 파이프라인 평가의 기준이다. 원본 PDF·기존 산출물은 수정하지 않는다.

각 코드 셀은 하나의 결과표만 표시한다. API 원문, 공통 TXT, 필드 예측 JSON, 지표는 모두 이 노트북의 실행 폴더에 새로 저장한다.


In [1]:
from __future__ import annotations
import base64, hashlib, html, json, os, re
from collections import Counter
from pathlib import Path
import fitz, pandas as pd
from dotenv import load_dotenv
from rapidfuzz.distance import Levenshtein

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'data' / 'raw').is_dir()), None)
if PROJECT_ROOT is None: raise RuntimeError(f'프로젝트 루트를 찾지 못했습니다: {cwd}')
SOURCE_RUN_ID = os.getenv('SOURCE_RUN_ID', '2026-07-25_live_ocr')  # 평가할 OCR 실행
OCR_RUN_ID = os.getenv('OCR_RUN_ID', '2026-07-26_live_ocr')  # 새 OCR 저장 실행 ID
EVALUATION_ID = 'evaluation_v4'
TARGETS = [('woori','Woori_Classic_EVERY_MILE_SKYPASS'),('shinhan','Shinhan_Toss_Mr.Life_20251231'),('samsung','Samsung_iD_ALL'),('lotte','Lotte_LOCA_LIKIT_Eat'),('kookmin','Kookmin_Friend_20210917')]
TARGET_ISSUER = os.getenv('OCR_TARGET_ISSUER')
if TARGET_ISSUER: TARGETS = [x for x in TARGETS if x[0] == TARGET_ISSUER]
RUN_OPENAI_VISION = False  # OpenAI Vision OCR 셀만 실행할 때 True
RUN_UPSTAGE = False  # Upstage OCR 셀만 실행할 때 True
OVERWRITE_OCR = False  # False면 이미 저장된 OCR 원문을 재호출하지 않음
OPENAI_VISION_MODEL = os.getenv('OPENAI_VISION_MODEL', 'gpt-5.4-mini')
UPSTAGE_MODEL = os.getenv('UPSTAGE_MODEL', 'document-parse')
RUN_FIELD_EXTRACTION = True
FIELD_EXTRACTION_MODEL = os.getenv('FIELD_EXTRACTION_MODEL', 'gpt-5.4-mini')
RUNS_ROOT = PROJECT_ROOT/'notebooks'/'data'/'07_goldset_vision_upstage_comparison'
SOURCE_ROOT = RUNS_ROOT/SOURCE_RUN_ID
OCR_RUN_ROOT = RUNS_ROOT/OCR_RUN_ID
OUTPUT_ROOT = SOURCE_ROOT/EVALUATION_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
load_dotenv(PROJECT_ROOT/'.env'); OPENAI_API_KEY=os.getenv('OPENAI_API_KEY'); UPSTAGE_API_KEY=os.getenv('UPSTAGE_API_KEY')
print({'source_run': SOURCE_RUN_ID, 'ocr_run': OCR_RUN_ID, 'evaluation': EVALUATION_ID, 'targets': [x[0] for x in TARGETS], 'run_openai_vision': RUN_OPENAI_VISION, 'run_upstage': RUN_UPSTAGE, 'field_extraction': RUN_FIELD_EXTRACTION})


{'source_run': '2026-07-25_live_ocr', 'evaluation': 'evaluation_v4', 'targets': ['woori', 'shinhan', 'samsung', 'lotte', 'kookmin'], 'field_extraction': True}


In [2]:
RAW_PAGE=re.compile(r'^\[page\s+(\d+)\]\s*$',re.I|re.M); PANEL=re.compile(r'^\[(?:left|center|right)_panel\]\s*$',re.I|re.M)
def read(p): return Path(p).read_text(encoding='utf-8')
def pages(text, marker=RAW_PAGE):
    found=list(marker.finditer(text)); return {int(m.group(1)):text[m.end():(found[i+1].start() if i+1<len(found) else len(text))].strip() for i,m in enumerate(found)}
def norm(text):
    text=PANEL.sub('',html.unescape(text or '')).lower(); text=re.sub(r'<[^>]+>',' ',text); text=re.sub(r'[^0-9a-z가-힣]+',' ',text); return re.sub(r'\s+',' ',text).strip()
def prf(reference, candidate):
    r,c=Counter(reference.split()),Counter(candidate.split()); hit=sum((r&c).values()); p=hit/max(sum(c.values()),1); q=hit/max(sum(r.values()),1); return p,q,2*p*q/max(p+q,1e-12)
def numeric_tokens(text): return Counter(re.findall(r'(?<![0-9])[0-9][0-9,.%~:/-]*(?![0-9])', norm(text)))
def get_upstage_pages(payload):
    out={}
    for e in payload.get('elements',[]):
        c=e.get('content',{}); out.setdefault(e.get('page'),[]).append(c.get('text') or c.get('markdown') or c.get('html') or '')
    return {k:'\n'.join(v) for k,v in out.items()}
def canonical(value):
    if isinstance(value,str): return norm(value)
    if isinstance(value,list): return [canonical(x) for x in value]
    if isinstance(value,dict): return {k:canonical(v) for k,v in sorted(value.items())}
    return value
def shape(value):
    if isinstance(value,dict): return {k:shape(v) for k,v in value.items()}
    if isinstance(value,list): return [shape(value[0])] if value else []
    return type(value).__name__
def field_schema(gold):
    return {'field_labels':[{'id':x['id'],'page_num':x['page_num'],'context_terms':x.get('context_terms',[]),'value_shape':shape(x['value'])} for x in gold.get('field_labels',[])], 'numeric_labels':[{'id':x['id'],'page_num':x['page_num'],'context_terms':x.get('context_terms',[]),'unit':x.get('unit'),'value_shape':{'surface_text':'str','normalized_value':type(x.get('normalized_value')).__name__}} for x in gold.get('numeric_labels',[])], 'table_labels':[{'id':x['id'],'page_num':x['page_num'],'column_count':len(x.get('headers',[]))} for x in gold.get('table_labels',[])]}


## OCR 원문 수집 (선택 실행)

두 OCR 셀은 독립적이다. 실행할 엔진의 플래그만 `True`로 바꾸고 해당 셀만 실행한다. 새 결과는 `OCR_RUN_ID` 아래에 저장되며, 기존 실행 폴더를 덮어쓰지 않는다. 두 엔진 수집을 마친 뒤 해당 실행을 평가하려면 `SOURCE_RUN_ID=OCR_RUN_ID`로 맞춘다.


In [ ]:
# OCR 수집 공통 함수: 이 셀은 외부 API를 호출하지 않는다.
def require_env(name, value):
    if not value: raise RuntimeError(f'{name} 환경변수가 없습니다. 프로젝트 루트 .env에만 설정하세요.')
    return value
def pdf_sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024), b''): digest.update(chunk)
    return digest.hexdigest()
def target_documents():
    docs=[]
    for issuer,card in TARGETS:
        pdf_path=PROJECT_ROOT/'data'/'raw'/issuer/f'{card}.pdf'
        if not pdf_path.is_file(): raise FileNotFoundError(f'원본 PDF가 없습니다: {pdf_path}')
        with fitz.open(pdf_path) as pdf: page_count=len(pdf)
        docs.append({'issuer':issuer,'card_name':card,'pdf_path':pdf_path,'page_count':page_count,'pdf_sha256':pdf_sha256(pdf_path)})
    return docs
def render_page_png(pdf_path, page_num):
    with fitz.open(pdf_path) as pdf:
        pixmap=pdf[page_num-1].get_pixmap(matrix=fitz.Matrix(2,2), alpha=False)
    return pixmap.tobytes('png')
def write_ocr_manifest(documents):
    OCR_RUN_ROOT.mkdir(parents=True, exist_ok=True)
    payload={'run_id':OCR_RUN_ID,'documents':[{**doc,'pdf_path':str(doc['pdf_path'])} for doc in documents], 'openai_vision_model':OPENAI_VISION_MODEL, 'upstage_model':UPSTAGE_MODEL}
    (OCR_RUN_ROOT/'ocr_manifest.json').write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf-8')


In [ ]:
# OCR 실행 1: OpenAI Vision만 수집한다. RUN_OPENAI_VISION=True일 때만 API를 호출한다.
if not RUN_OPENAI_VISION:
    print('SKIPPED: RUN_OPENAI_VISION=False. Upstage 셀과 독립적으로 실행할 수 있습니다.')
else:
    from openai import OpenAI
    documents=target_documents(); client=OpenAI(api_key=require_env('OPENAI_API_KEY', OPENAI_API_KEY)); rows=[]
    for doc in documents:
        for page_num in range(1,doc['page_count']+1):
            output_path=OCR_RUN_ROOT/'raw'/'vision'/doc['issuer']/doc['card_name']/f'{page_num:03d}.json'
            if output_path.exists() and not OVERWRITE_OCR:
                status='cached'
            else:
                image=base64.b64encode(render_page_png(doc['pdf_path'],page_num)).decode('ascii')
                response=client.responses.create(model=OPENAI_VISION_MODEL,input=[{'role':'user','content':[{'type':'input_text','text':'카드 안내 PDF 페이지의 모든 텍스트를 읽기 순서대로 전사하세요. 표는 Markdown 표로 작성하고, 요약이나 추론은 하지 마세요.'},{'type':'input_image','image_url':f'data:image/png;base64,{image}','detail':'high'}]}])
                output_path.parent.mkdir(parents=True,exist_ok=True)
                output_path.write_text(json.dumps({'model':OPENAI_VISION_MODEL,'page_text':response.output_text.strip()},ensure_ascii=False,indent=2),encoding='utf-8')
                status='created'
            rows.append({'issuer':doc['issuer'],'card_name':doc['card_name'],'page_num':page_num,'status':status,'path':str(output_path)})
    write_ocr_manifest(documents)
    openai_ocr_manifest=pd.DataFrame(rows); display(openai_ocr_manifest)


In [ ]:
# OCR 실행 2: Upstage만 수집한다. RUN_UPSTAGE=True일 때만 API를 호출한다.
if not RUN_UPSTAGE:
    print('SKIPPED: RUN_UPSTAGE=False. OpenAI Vision 셀과 독립적으로 실행할 수 있습니다.')
else:
    import requests
    documents=target_documents(); rows=[]
    for doc in documents:
        output_path=OCR_RUN_ROOT/'raw'/'upstage'/doc['issuer']/f"{doc['card_name']}.json"
        if output_path.exists() and not OVERWRITE_OCR:
            status='cached'
        else:
            with doc['pdf_path'].open('rb') as stream:
                response=requests.post('https://api.upstage.ai/v1/document-digitization',headers={'Authorization':f"Bearer {require_env('UPSTAGE_API_KEY', UPSTAGE_API_KEY)}"},files={'document':stream},data={'model':UPSTAGE_MODEL,'ocr':'force'},timeout=180)
            response.raise_for_status(); output_path.parent.mkdir(parents=True,exist_ok=True)
            output_path.write_text(json.dumps(response.json(),ensure_ascii=False,indent=2),encoding='utf-8')
            status='created'
        rows.append({'issuer':doc['issuer'],'card_name':doc['card_name'],'status':status,'path':str(output_path)})
    write_ocr_manifest(documents)
    upstage_ocr_manifest=pd.DataFrame(rows); display(upstage_ocr_manifest)


In [3]:
# 결과물 1: provider 원문에서 추출한 공통 TXT 보존본
manifest=[]
for issuer,card in TARGETS:
    vision_dir=SOURCE_ROOT/'raw'/'vision'/issuer/card
    vision={int(p.stem):json.loads(read(p))['page_text'] for p in sorted(vision_dir.glob('*.json'))}
    upstage=get_upstage_pages(json.loads(read(SOURCE_ROOT/'raw'/'upstage'/issuer/f'{card}.json')))
    for engine,content in [('openai',vision),('upstage',upstage)]:
        path=OUTPUT_ROOT/'text'/engine/issuer/f'{card}.txt'; path.parent.mkdir(parents=True,exist_ok=True)
        path.write_text('\n\n'.join(f'[page {n}]\n{content.get(n,"")}' for n in sorted(content)),encoding='utf-8')
        manifest.append({'issuer':issuer,'card_name':card,'engine':engine,'pages':len(content),'text_path':str(path)})
text_manifest=pd.DataFrame(manifest); text_manifest.to_csv(OUTPUT_ROOT/'text_manifest.csv',index=False); display(text_manifest)


,issuer,card_name,engine,pages,text_path
0,woori,Woori_Classic_EVERY_MILE_SKYPASS,openai,2,/home/sms/openclaw_file/PickCardU/notebooks/da...
1,woori,Woori_Classic_EVERY_MILE_SKYPASS,upstage,2,/home/sms/openclaw_file/PickCardU/notebooks/da...
2,shinhan,Shinhan_Toss_Mr.Life_20251231,openai,2,/home/sms/openclaw_file/PickCardU/notebooks/da...
3,shinhan,Shinhan_Toss_Mr.Life_20251231,upstage,2,/home/sms/openclaw_file/PickCardU/notebooks/da...
4,samsung,Samsung_iD_ALL,openai,8,/home/sms/openclaw_file/PickCardU/notebooks/da...
5,samsung,Samsung_iD_ALL,upstage,8,/home/sms/openclaw_file/PickCardU/notebooks/da...
6,lotte,Lotte_LOCA_LIKIT_Eat,openai,8,/home/sms/openclaw_file/PickCardU/notebooks/da...
7,lotte,Lotte_LOCA_LIKIT_Eat,upstage,8,/home/sms/openclaw_file/PickCardU/notebooks/da...
8,kookmin,Kookmin_Friend_20210917,openai,2,/home/sms/openclaw_file/PickCardU/notebooks/da...
9,kookmin,Kookmin_Friend_20210917,upstage,2,/home/sms/openclaw_file/PickCardU/notebooks/da...


In [4]:
# 결과물 2: gold/raw 대비 전체 OCR 성능 (페이지 단위)
rows=[]
for issuer,card in TARGETS:
    gold=pages(read(PROJECT_ROOT/'data'/'ocr_benchmark'/'gold'/'raw'/issuer/f'{card}.txt'))
    for engine in ['openai','upstage']:
        predicted=pages(read(OUTPUT_ROOT/'text'/engine/issuer/f'{card}.txt'))
        for page_num,truth in gold.items():
            a,b=norm(truth),norm(predicted.get(page_num,'')); p,r,f=prf(a,b); np,nr,nf=prf(' '.join(numeric_tokens(a).elements()),' '.join(numeric_tokens(b).elements()))
            rows.append({'issuer':issuer,'card_name':card,'page_num':page_num,'engine':engine,'cer':Levenshtein.distance(a,b)/max(len(a),1),'token_precision':p,'token_recall':r,'token_f1':f,'numeric_f1':nf,'exact_text_match':a==b})
full_page=pd.DataFrame(rows); full_page.to_csv(OUTPUT_ROOT/'full_page_metrics.csv',index=False)
display(full_page.groupby('engine')[['cer','token_precision','token_recall','token_f1','numeric_f1','exact_text_match']].mean().round(4))


,cer,token_precision,token_recall,token_f1,numeric_f1,exact_text_match
engine,,,,,,
openai,0.6242,0.8626,0.9492,0.8826,0.8128,0.0000
upstage,0.5675,0.9039,0.9804,0.9195,0.8402,0.0909


In [5]:
# 결과물 3: OpenAI↔Upstage 전체 텍스트 교차검증과 gold 기준 판정
engine_rows=[]
for (issuer,card,page_num),group in full_page.groupby(['issuer','card_name','page_num']):
    texts={engine:norm(pages(read(OUTPUT_ROOT/'text'/engine/issuer/f'{card}.txt')).get(page_num,'')) for engine in ['openai','upstage']}
    _,_,agreement_f1=prf(texts['openai'],texts['upstage']); _,_,numeric_agreement=prf(' '.join(numeric_tokens(texts['openai']).elements()),' '.join(numeric_tokens(texts['upstage']).elements()))
    openai=group.loc[group.engine=='openai'].iloc[0]; upstage=group.loc[group.engine=='upstage'].iloc[0]
    candidate=agreement_f1>=0.98 and numeric_agreement==1.0
    engine_rows.append({'issuer':issuer,'card_name':card,'page_num':page_num,'auto_pass_candidate':candidate,'engine_text_f1':agreement_f1,'engine_numeric_f1':numeric_agreement,'openai_cer':openai.cer,'upstage_cer':upstage.cer,'both_exact_vs_gold':bool(openai.exact_text_match and upstage.exact_text_match),'openai_only_exact':bool(openai.exact_text_match and not upstage.exact_text_match),'upstage_only_exact':bool(upstage.exact_text_match and not openai.exact_text_match),'both_not_exact':bool(not openai.exact_text_match and not upstage.exact_text_match)})
engine_full=pd.DataFrame(engine_rows); engine_full.to_csv(OUTPUT_ROOT/'full_engine_validation.csv',index=False)
display(engine_full[['auto_pass_candidate','both_exact_vs_gold','openai_only_exact','upstage_only_exact','both_not_exact']].mean().rename('rate').to_frame().round(4))


,rate
auto_pass_candidate,0.3182
both_exact_vs_gold,0.0000
openai_only_exact,0.0000
upstage_only_exact,0.0909
both_not_exact,0.9091


In [6]:
# 결과물 4: OCR TXT를 공통 필드 JSON으로 변환. gold 값은 프롬프트에 전달하지 않는다.
def nullable(schema): return {'anyOf':[schema, {'type':'null'}]}
def value_schema(shape):
    if isinstance(shape,dict):
        return nullable({'type':'object','properties':{k:value_schema(v) for k,v in shape.items()},'required':list(shape),'additionalProperties':False})
    if isinstance(shape,list): return nullable({'type':'array','items':value_schema(shape[0]) if shape else {}})
    types={'str':'string','int':'number','float':'number','bool':'boolean'}
    return nullable({'type':types.get(shape,'string')})
def prediction_json_schema(schema):
    field_props={x['id']:value_schema(x['value_shape']) for x in schema['field_labels']}
    numeric_value=nullable({'type':'object','properties':{'surface_text':nullable({'type':'string'}),'normalized_value':nullable({'anyOf':[{'type':'string'},{'type':'number'}]})},'required':['surface_text','normalized_value'],'additionalProperties':False})
    numeric_props={x['id']:numeric_value for x in schema['numeric_labels']}
    cell=nullable({'anyOf':[{'type':'string'},{'type':'number'}]})
    table_value=nullable({'type':'object','properties':{'headers':{'type':'array','items':cell},'rows':{'type':'array','items':{'type':'array','items':cell}}},'required':['headers','rows'],'additionalProperties':False})
    table_props={x['id']:table_value for x in schema['table_labels']}
    return {'type':'object','properties':{'field_labels':{'type':'object','properties':field_props,'required':list(field_props),'additionalProperties':False},'numeric_labels':{'type':'object','properties':numeric_props,'required':list(numeric_props),'additionalProperties':False},'table_labels':{'type':'object','properties':table_props,'required':list(table_props),'additionalProperties':False}},'required':['field_labels','numeric_labels','table_labels'],'additionalProperties':False}
def extract_structured(ocr_text, schema):
    if not OPENAI_API_KEY: raise RuntimeError('OPENAI_API_KEY가 없습니다.')
    prompt='OCR 텍스트만 근거로 스키마의 각 필드를 추출하세요. OCR에 없는 값은 null로 두고 추정·보정하지 마세요. 원문 표기는 유지하고 normalized_value만 숫자 또는 날짜 같은 정규값으로 쓰세요. 표는 headers와 rows를 반환하세요. JSON 구조와 모든 field_id는 제공된 JSON Schema를 반드시 따르세요.'+'\n\nVALUE-LESS SCHEMA:\n'+json.dumps(schema,ensure_ascii=False)+'\n\nOCR TEXT:\n'+ocr_text
    from openai import OpenAI
    response=OpenAI(api_key=OPENAI_API_KEY).responses.create(model=FIELD_EXTRACTION_MODEL,input=prompt,text={'format':{'type':'json_schema','name':'ocr_field_prediction','strict':True,'schema':prediction_json_schema(schema)}},store=False)
    return json.loads(response.output_text)

field_manifest=[]
for issuer,card in TARGETS:
    gold=json.loads(read(PROJECT_ROOT/'data'/'ocr_benchmark'/'gold'/'structured'/issuer/f'{card}.json')); schema=field_schema(gold)
    for engine in ['openai','upstage']:
        path=OUTPUT_ROOT/'structured'/engine/issuer/f'{card}.json'; path.parent.mkdir(parents=True,exist_ok=True)
        if RUN_FIELD_EXTRACTION and not path.exists(): path.write_text(json.dumps(extract_structured(read(OUTPUT_ROOT/'text'/engine/issuer/f'{card}.txt'),schema),ensure_ascii=False,indent=2),encoding='utf-8')
        field_manifest.append({'issuer':issuer,'card_name':card,'engine':engine,'prediction_path':str(path),'exists':path.exists()})
field_manifest=pd.DataFrame(field_manifest); field_manifest.to_csv(OUTPUT_ROOT/'structured_manifest.csv',index=False); display(field_manifest)


,issuer,card_name,engine,prediction_path,exists
0,woori,Woori_Classic_EVERY_MILE_SKYPASS,openai,/home/sms/openclaw_file/PickCardU/notebooks/da...,True
1,woori,Woori_Classic_EVERY_MILE_SKYPASS,upstage,/home/sms/openclaw_file/PickCardU/notebooks/da...,True
2,shinhan,Shinhan_Toss_Mr.Life_20251231,openai,/home/sms/openclaw_file/PickCardU/notebooks/da...,True
3,shinhan,Shinhan_Toss_Mr.Life_20251231,upstage,/home/sms/openclaw_file/PickCardU/notebooks/da...,True
4,samsung,Samsung_iD_ALL,openai,/home/sms/openclaw_file/PickCardU/notebooks/da...,True
5,samsung,Samsung_iD_ALL,upstage,/home/sms/openclaw_file/PickCardU/notebooks/da...,True
6,lotte,Lotte_LOCA_LIKIT_Eat,openai,/home/sms/openclaw_file/PickCardU/notebooks/da...,True
7,lotte,Lotte_LOCA_LIKIT_Eat,upstage,/home/sms/openclaw_file/PickCardU/notebooks/da...,True
8,kookmin,Kookmin_Friend_20210917,openai,/home/sms/openclaw_file/PickCardU/notebooks/da...,True
9,kookmin,Kookmin_Friend_20210917,upstage,/home/sms/openclaw_file/PickCardU/notebooks/da...,True


In [7]:
# 결과물 5: gold/structured 대비 필드 exact-match 평가
records=[]
for issuer,card in TARGETS:
    gold=json.loads(read(PROJECT_ROOT/'data'/'ocr_benchmark'/'gold'/'structured'/issuer/f'{card}.json'))
    gold_sets={'field_labels':{x['id']:x['value'] for x in gold.get('field_labels',[])},'numeric_labels':{x['id']:{'surface_text':x['surface_text'],'normalized_value':x['normalized_value']} for x in gold.get('numeric_labels',[])},'table_labels':{x['id']:{'headers':x['headers'],'rows':x['rows']} for x in gold.get('table_labels',[])}}
    critical={x['id'] for x in gold.get('field_labels',[]) if x.get('critical')}|{x['id'] for x in gold.get('numeric_labels',[]) if x.get('critical')}
    for engine in ['openai','upstage']:
        pred=json.loads(read(OUTPUT_ROOT/'structured'/engine/issuer/f'{card}.json'))
        for kind,values in gold_sets.items():
            got=pred.get(kind,{})
            for field_id,truth in values.items():
                value=got.get(field_id); correct=canonical(value)==canonical(truth); present=value is not None
                records.append({'issuer':issuer,'card_name':card,'engine':engine,'kind':kind,'field_id':field_id,'critical':field_id in critical,'present':present,'exact_match':correct})
field_metrics=pd.DataFrame(records); field_metrics.to_csv(OUTPUT_ROOT/'field_exact_metrics.csv',index=False)
display(field_metrics.groupby('engine').agg(field_count=('field_id','size'),value_precision=('exact_match',lambda x:x.sum()/max(field_metrics.loc[x.index,'present'].sum(),1)),value_recall=('exact_match','mean'),critical_exact=('exact_match',lambda x:x[field_metrics.loc[x.index,'critical']].mean())).round(4))


,field_count,value_precision,value_recall,critical_exact
engine,,,,
openai,117,0.3932,0.3932,0.4318
upstage,117,0.3932,0.3932,0.4205


In [8]:
# 결과물 6: OpenAI↔Upstage 필드 교차검증과 gold 기준 4분류
pairs=[]
for keys,group in field_metrics.groupby(['issuer','card_name','kind','field_id']):
    o=group.loc[group.engine=='openai'].iloc[0]; u=group.loc[group.engine=='upstage'].iloc[0]
    pairs.append({'issuer':keys[0],'card_name':keys[1],'kind':keys[2],'field_id':keys[3],'engines_agree':canonical(json.loads(read(OUTPUT_ROOT/'structured'/'openai'/keys[0]/f'{keys[1]}.json')).get(keys[2],{}).get(keys[3]))==canonical(json.loads(read(OUTPUT_ROOT/'structured'/'upstage'/keys[0]/f'{keys[1]}.json')).get(keys[2],{}).get(keys[3])),'both_correct':bool(o.exact_match and u.exact_match),'openai_only_correct':bool(o.exact_match and not u.exact_match),'upstage_only_correct':bool(u.exact_match and not o.exact_match),'both_wrong':bool(not o.exact_match and not u.exact_match)})
field_engine=pd.DataFrame(pairs); field_engine.to_csv(OUTPUT_ROOT/'field_engine_validation.csv',index=False)
display(field_engine[['engines_agree','both_correct','openai_only_correct','upstage_only_correct','both_wrong']].mean().rename('rate').to_frame().round(4))


,rate
engines_agree,0.7863
both_correct,0.3590
openai_only_correct,0.0342
upstage_only_correct,0.0342
both_wrong,0.5726


In [9]:
# 결과물 7: 실행 메타데이터와 모든 평가 산출물 경로
summary={'source_run_id':SOURCE_RUN_ID,'evaluation_id':EVALUATION_ID,'targets':TARGETS,'full_page_metrics':'full_page_metrics.csv','full_engine_validation':'full_engine_validation.csv','field_metrics':'field_exact_metrics.csv','field_engine_validation':'field_engine_validation.csv','text_manifest':'text_manifest.csv','structured_manifest':'structured_manifest.csv'}
(OUTPUT_ROOT/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame([summary]))


,source_run_id,evaluation_id,targets,full_page_metrics,full_engine_validation,field_metrics,field_engine_validation,text_manifest,structured_manifest
0,2026-07-25_live_ocr,evaluation_v4,"[(woori, Woori_Classic_EVERY_MILE_SKYPASS), (s...",full_page_metrics.csv,full_engine_validation.csv,field_exact_metrics.csv,field_engine_validation.csv,text_manifest.csv,structured_manifest.csv
